# bagel-space
Use one A100 80 GB GPU. Publish this repository to GitHub before using the clone cell.


## 1. Setup
Run once in a fresh runtime; restart the session after installation.


In [ ]:
!nvidia-smi
!git clone https://github.com/Nicerova7/bagel-space.git /content/bagel-space
!git clone https://github.com/ByteDance-Seed/Bagel.git /content/BAGEL
!git -C /content/BAGEL checkout a2fa77dd8caeefc41e6607ae0ec17408d3f4ee9f
%pip install -r /content/bagel-space/requirements.txt
%env MAX_JOBS=4
%pip install flash_attn==2.5.8 --no-build-isolation
print("Restart the session, then continue below.")


### Load the model
Run after restarting; later cells reuse this adapter.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/bagel-space")
UPSTREAM = Path("/content/BAGEL")
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(UPSTREAM) + os.pathsep + str(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM)]

from bagel_space.adapter import BagelAdapter
from bagel_space.capture import Capture

adapter = BagelAdapter()
print(torch.cuda.get_device_name(0), adapter.dtype)
print("Layers:", len(adapter.layers), "| Hidden size:", adapter.model.hidden_size)


## 2. Choose an image
Use the official example or upload your own image below.


In [ ]:
images = PROJECT / "inputs/images"
images.mkdir(parents=True, exist_ok=True)
shutil.copy2(UPSTREAM / "test_images/meme.jpg", images / "example.jpg")
image_path = images / "example.jpg"
display(adapter.prepare_understanding(image_path))


### Optional: upload an image


In [ ]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    image_path = images / Path(name).name
    image_path.write_bytes(data)
print("Image:", image_path)


## 3. Understanding and hidden states
Capture image and question prefills separately, before decoding an answer.


In [ ]:
question = "Describe this image."
layers = [0, 8, 16]
seed = 42

adapter.seed(seed)
prepared_image = adapter.prepare_understanding(image_path)
with Capture(adapter.layers, layers, "image") as visual, Capture(
    adapter.layers, layers, "text",
) as text:
    context = adapter.forward_understanding(prepared_image, question)

understanding_output = PROJECT / "outputs/understanding"
understanding_output.mkdir(parents=True, exist_ok=True)
understanding_metadata = dict(
    **adapter.metadata(), image=str(image_path), question=question,
    image_size=list(prepared_image.size), layers=layers, seed=seed,
    alignment=dict(image=visual.alignment, text=text.alignment),
    text_token_ids=[adapter.token_ids["bos_token_id"]]
    + adapter.tokenizer.encode(question) + [adapter.token_ids["eos_token_id"]],
)
torch.save(
    dict(hidden_states=dict(image=visual.states, text=text.states), metadata=understanding_metadata),
    understanding_output / "hidden_states.pt",
)
(understanding_output / "metadata.json").write_text(
    json.dumps(understanding_metadata, indent=2), encoding="utf-8",
)
for stage, states in (("image", visual.states), ("text", text.states)):
    for layer, hidden in states.items():
        print(stage, "| Layer:", layer, "| Shape:", tuple(hidden.shape))


### Optional: generate an answer
Reuses the captured context; no capture hooks remain active.


In [ ]:
answer = adapter.answer(context, max_new_tokens=128)
(understanding_output / "answer.txt").write_text(answer + "\n", encoding="utf-8")
print(answer)


## 4. Generation and hidden states
Capture the first conditional flow evaluation, not the cached prompt or final image.


In [ ]:
prompt = "A photograph of a golden retriever sitting on green grass."
steps = 50
guidance = 1.0
seed = 42
layers = [0, 8, 16]

with Capture(adapter.layers, layers, "generation") as captured:
    image = adapter.generate(prompt, steps=steps, guidance=guidance, seed=seed)

generation_output = PROJECT / "outputs/generation"
generation_output.mkdir(parents=True, exist_ok=True)
image.save(generation_output / "image.png")
generation_metadata = dict(
    **adapter.metadata(), prompt=prompt, resolution=1024, layers=layers,
    steps=steps, guidance=guidance, seed=seed, image_guidance=1.0,
    cfg_interval=[0.4, 1.0], timestep_shift=3.0,
    cfg_renorm_min=0.0, cfg_renorm_type="global", taylorseer=False,
    evaluation="first conditional flow evaluation", alignment=captured.alignment,
)
torch.save(
    dict(hidden_states=captured.states, metadata=generation_metadata),
    generation_output / "hidden_states.pt",
)
(generation_output / "metadata.json").write_text(
    json.dumps(generation_metadata, indent=2), encoding="utf-8",
)
for layer, hidden in captured.states.items():
    print("Layer:", layer, "| Shape:", tuple(hidden.shape))
display(image)


## 5. Download outputs


In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/bagel-space-outputs", "zip", PROJECT / "outputs")
files.download(archive)
